# 04 — Synthetic ground-truth Yamada recovery

This notebook validates the complete controlled pipeline

\[
G_{\rm true}
\longrightarrow
G_\lambda
\longrightarrow
V_{\lambda,N}
\longrightarrow
\widehat G
\longrightarrow
\Upsilon(\widehat G;A).
\]

The benchmark is deliberately restricted to **connected trivalent spatial graphs**.
This avoids a known projection pathology of representing closed knot/link components
as degree-2 self-loops: under some rigid rotations those artificial self-loop
representations can generate many spurious projected crossings. Knot/link-only
examples therefore do not enter this headline graph-spine recovery test.

Every deformation is an orientation-preserving invertible affine map
\(F(x)=Mx+b\) with \(\det M>0\), so the generating spatial graph is preserved up to
ambient isotopy before voxelization.

The headline success condition is

\[
\Upsilon(\widehat G;A)=\Upsilon(G_{\rm true};A).
\]

`QUICK_MODE=True` is the small correctness run used by CI. Set it to `False`
for the \(N=150,\ldots,300\) local benchmark.


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
import hashlib
import json
import sys

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if not (ROOT / "src" / "knotted_graph").exists():
    raise RuntimeError("Run this notebook from inside the KnottedGraph checkout.")

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import sympy as sp
from skimage.morphology import ball, dilation, skeletonize

import knotted_graph
from knotted_graph.core import simplify_edges
from knotted_graph.extraction import skeleton_image_to_graph
from knotted_graph.projection import compute_yamada_polynomial

kg_path = Path(knotted_graph.__file__).resolve()
if SRC not in kg_path.parents:
    raise RuntimeError(f"A stale knotted_graph was imported from {kg_path}")

A = sp.Symbol("A")
BOUND = 1.35
PROJECTION_SAMPLES = 16

# Small CI/local smoke test by default.
QUICK_MODE = True

if QUICK_MODE:
    RESOLUTIONS = [80]
    TUBE_RADII_VOX = [1]
    TRANSFORMS = ["identity", "rotate", "affine"]
    ACTIVE_CASE_NAMES = [
        "theta3_planar",
        "K4",
        "triangular_prism",
    ]
else:
    # Publication-scale local sweep requested for the full benchmark.
    RESOLUTIONS = list(range(150, 301, 25))  # 150, 175, ..., 300
    TUBE_RADII_VOX = [1, 2, 3]
    TRANSFORMS = ["identity", "rotate", "affine"]
    ACTIVE_CASE_NAMES = None  # use the full suite below

CACHE_SCHEMA = "trivalent-ground-truth-v3"
RESUME = True
CHECKPOINT = (
    ROOT
    / "User_guide"
    / "benchmarks"
    / "synthetic_ground_truth_results_v3.jsonl"
)

print("branch-local KnottedGraph:", kg_path)
print("mode:", "QUICK" if QUICK_MODE else "FULL")
print("resolutions:", RESOLUTIONS)
print("checkpoint:", CHECKPOINT)


## Ground-truth suite

The full suite contains only connected, bridgeless, exactly trivalent graphs.
It combines multigraph and simple-graph families:

- planar and bowed \(\Theta_3\);
- \(K_4\);
- triangular, square/cube, and pentagonal prisms;
- the dodecahedral graph;
- the Frucht graph when a planar embedding is available;
- \(K_{3,3}\);
- Petersen;
- Heawood.

Planar abstract graphs use deterministic planar layouts. Nonplanar graphs use
deterministic generic 3-D spring embeddings and require a positive nonincident-edge
clearance before entering the suite.


In [ ]:
@dataclass
class Case:
    name: str
    graph: nx.MultiGraph
    radius_cap: float


def embedded_graph(nodes, edges):
    G = nx.MultiGraph()
    for node, pos in nodes.items():
        G.add_node(node, pos=np.asarray(pos, dtype=float))
    for u, v, pts in edges:
        G.add_edge(u, v, pts=np.asarray(pts, dtype=float))
    return G


def normalize_positions(P, scale=0.72):
    P = np.asarray(P, dtype=float)
    P = P - P.mean(axis=0)
    radius = np.max(np.linalg.norm(P, axis=1))
    if radius <= 0:
        raise ValueError("degenerate embedding")
    return P * (scale / radius)


def theta_case(name, bowed=False, n=500):
    t = np.linspace(0.0, 1.0, n)
    x = -0.72 + 1.44 * t

    if not bowed:
        curves = [
            np.c_[x, -0.58*np.sin(np.pi*t), np.zeros_like(t)],
            np.c_[x, np.zeros_like(t), np.zeros_like(t)],
            np.c_[x,  0.58*np.sin(np.pi*t), np.zeros_like(t)],
        ]
    else:
        curves = [
            np.c_[x, -0.58*np.sin(np.pi*t),  0.16*np.sin(2*np.pi*t)],
            np.c_[x,  0.10*np.sin(2*np.pi*t), -0.10*np.sin(np.pi*t)],
            np.c_[x,  0.58*np.sin(np.pi*t), -0.16*np.sin(2*np.pi*t)],
        ]

    for P in curves:
        P[0] = [-0.72, 0.0, 0.0]
        P[-1] = [0.72, 0.0, 0.0]

    return Case(
        name,
        embedded_graph(
            {"u": curves[0][0], "v": curves[0][-1]},
            [("u", "v", P) for P in curves],
        ),
        radius_cap=0.060,
    )


def segment_distance(p1, q1, p2, q2):
    u = q1 - p1
    v = q2 - p2
    w = p1 - p2
    a = u @ u
    b = u @ v
    c = v @ v
    d = u @ w
    e = v @ w
    D = a*c - b*b

    if D < 1e-14:
        s = 0.0
        t = np.clip(e/c if c > 1e-14 else 0.0, 0.0, 1.0)
    else:
        s = np.clip((b*e - c*d)/D, 0.0, 1.0)
        t = np.clip((a*e - b*d)/D, 0.0, 1.0)

    if a > 1e-14:
        s = np.clip((b*t - d)/a, 0.0, 1.0)
    if c > 1e-14:
        t = np.clip((b*s + e)/c, 0.0, 1.0)

    return float(np.linalg.norm(w + s*u - t*v))


def straight_edge_clearance(G, positions):
    edges = list(G.edges())
    best = np.inf
    for i, (u, v) in enumerate(edges):
        for a, b in edges[i+1:]:
            if {u, v} & {a, b}:
                continue
            best = min(
                best,
                segment_distance(
                    positions[u], positions[v],
                    positions[a], positions[b],
                ),
            )
    return best


def cubic_case(name, abstract_graph, *, planar, seed, cap=0.040):
    G = nx.Graph(abstract_graph)
    if not nx.is_connected(G):
        raise ValueError(f"{name} must be connected")
    if not all(degree == 3 for _, degree in G.degree()):
        raise ValueError(f"{name} must be exactly trivalent")
    if list(nx.bridges(G)):
        raise ValueError(f"{name} must be bridgeless")

    if planar:
        is_planar, _ = nx.check_planarity(G)
        if not is_planar:
            raise ValueError(f"{name} was declared planar but is not planar")
        layout = nx.planar_layout(G)
        X = normalize_positions(
            np.array(
                [[layout[node][0], layout[node][1], 0.0] for node in G],
                dtype=float,
            ),
            scale=0.72,
        )
        positions = {node: X[i] for i, node in enumerate(G)}
    else:
        positions = None
        for trial in range(200):
            layout = nx.spring_layout(
                G,
                dim=3,
                seed=seed + trial,
                iterations=500,
                scale=1.0,
            )
            X = normalize_positions(
                np.array([layout[node] for node in G], dtype=float),
                scale=0.72,
            )
            trial_positions = {node: X[i] for i, node in enumerate(G)}
            if straight_edge_clearance(G, trial_positions) > 0.055:
                positions = trial_positions
                break
        if positions is None:
            raise RuntimeError(f"could not construct a clear 3-D embedding for {name}")

    H = embedded_graph(
        positions,
        [
            (u, v, np.linspace(positions[u], positions[v], 80))
            for u, v in G.edges()
        ],
    )
    return Case(name, H, radius_cap=cap)


def maybe_planar_case(name, abstract_graph, seed, cap=0.040):
    is_planar, _ = nx.check_planarity(nx.Graph(abstract_graph))
    return cubic_case(
        name,
        abstract_graph,
        planar=is_planar,
        seed=seed,
        cap=cap,
    )


ALL_CASES = [
    theta_case("theta3_planar", bowed=False),
    theta_case("theta3_bowed", bowed=True),
    cubic_case("K4", nx.complete_graph(4), planar=True, seed=11, cap=0.052),
    cubic_case(
        "triangular_prism",
        nx.circular_ladder_graph(3),
        planar=True,
        seed=12,
        cap=0.045,
    ),
    cubic_case("cube", nx.cubical_graph(), planar=True, seed=13, cap=0.042),
    cubic_case(
        "pentagonal_prism",
        nx.circular_ladder_graph(5),
        planar=True,
        seed=14,
        cap=0.035,
    ),
    cubic_case(
        "dodecahedral",
        nx.dodecahedral_graph(),
        planar=True,
        seed=15,
        cap=0.027,
    ),
    maybe_planar_case("frucht", nx.frucht_graph(), seed=16, cap=0.030),
    cubic_case(
        "K3_3",
        nx.complete_bipartite_graph(3, 3),
        planar=False,
        seed=17,
        cap=0.032,
    ),
    cubic_case("petersen", nx.petersen_graph(), planar=False, seed=18, cap=0.030),
    cubic_case("heawood", nx.heawood_graph(), planar=False, seed=19, cap=0.024),
]

if ACTIVE_CASE_NAMES is None:
    CASES = ALL_CASES
else:
    by_name = {case.name: case for case in ALL_CASES}
    CASES = [by_name[name] for name in ACTIVE_CASE_NAMES]

for case in ALL_CASES:
    degrees = sorted(dict(case.graph.degree()).values())
    assert degrees and all(d == 3 for d in degrees), (case.name, degrees)
    assert nx.is_connected(nx.Graph(case.graph)), case.name

print("full suite:")
for case in ALL_CASES:
    print(
        f"  {case.name:20s} "
        f"V={case.graph.number_of_nodes():2d} "
        f"E={case.graph.number_of_edges():2d}"
    )
print("active cases:", [case.name for case in CASES])


In [ ]:
def Rxyz(a, b, c):
    a, b, c = np.deg2rad([a, b, c])
    Rx = np.array([
        [1, 0, 0],
        [0, np.cos(a), -np.sin(a)],
        [0, np.sin(a),  np.cos(a)],
    ])
    Ry = np.array([
        [ np.cos(b), 0, np.sin(b)],
        [0, 1, 0],
        [-np.sin(b), 0, np.cos(b)],
    ])
    Rz = np.array([
        [np.cos(c), -np.sin(c), 0],
        [np.sin(c),  np.cos(c), 0],
        [0, 0, 1],
    ])
    return Rz @ Ry @ Rx


def affine_transform(name):
    if name == "identity":
        M = np.eye(3)
        b = np.zeros(3)
    elif name == "rotate":
        M = Rxyz(21, 34, 13)
        b = np.array([0.04, -0.03, 0.02])
    elif name == "affine":
        M = (
            Rxyz(17, -23, 31)
            @ np.diag([1.08, 0.91, 1.03])
            @ np.array([
                [1.00, 0.13, 0.00],
                [0.00, 1.00, 0.09],
                [0.05, 0.00, 1.00],
            ])
        )
        b = np.array([-0.03, 0.04, -0.02])
    else:
        raise KeyError(name)

    determinant = float(np.linalg.det(M))
    if determinant <= 0:
        raise ValueError(f"{name}: transform is not orientation-preserving")
    return M, b


def deform(G, name):
    M, b = affine_transform(name)
    H = nx.MultiGraph()
    for node, data in G.nodes(data=True):
        H.add_node(node, pos=np.asarray(data["pos"]) @ M.T + b)
    for u, v, key, data in G.edges(keys=True, data=True):
        H.add_edge(
            u,
            v,
            pts=np.asarray(data["pts"]) @ M.T + b,
        )
    return H


In [ ]:
def _trimmed(P, fraction):
    if fraction <= 0:
        return P
    n = max(1, int(round(fraction * len(P))))
    if 2*n >= len(P):
        return P
    return P[n:-n]


def interior_separation(G, incident_trim=0.15):
    edge_records = [
        (u, v, np.asarray(data["pts"], dtype=float))
        for u, v, key, data in G.edges(keys=True, data=True)
    ]
    if len(edge_records) < 2:
        return np.inf

    best = np.inf
    for i, (u, v, P0) in enumerate(edge_records):
        for a, b, Q0 in edge_records[i+1:]:
            # IMPORTANT: never mutate P0/Q0 across pair comparisons.
            P = P0
            Q = Q0
            if {u, v} & {a, b}:
                P = _trimmed(P0, incident_trim)
                Q = _trimmed(Q0, incident_trim)

            # Chunk only the first array to bound temporary memory.
            for start in range(0, len(P), 128):
                block = P[start:start+128]
                distances2 = np.sum(
                    (block[:, None, :] - Q[None, :, :])**2,
                    axis=-1,
                )
                best = min(best, float(np.sqrt(distances2.min())))
    return best


def admissible(case, G, N, radius_vox):
    dx = 2*BOUND/(N-1)
    radius_world = radius_vox * dx
    separation = interior_separation(G)
    geometric_limit = (
        np.inf
        if not np.isfinite(separation)
        else 0.40 * separation
    )
    limit = min(case.radius_cap, geometric_limit)
    return radius_world <= limit, radius_world, separation, limit


def resample_polyline(P, step):
    out = []
    for p, q in zip(P[:-1], P[1:]):
        n = max(2, int(np.ceil(np.linalg.norm(q-p)/step)) + 1)
        out.append(np.linspace(p, q, n, endpoint=False))
    out.append(P[-1:])
    return np.vstack(out)


def voxelize(G, N, radius_vox):
    volume = np.zeros((N, N, N), dtype=bool)
    dx = 2*BOUND/(N-1)

    for u, v, key, data in G.edges(keys=True, data=True):
        P = resample_polyline(
            np.asarray(data["pts"], dtype=float),
            dx/3,
        )
        index = np.rint(
            (P + BOUND) / (2*BOUND) * (N-1)
        ).astype(int)
        index = np.clip(index, 0, N-1)
        volume[index[:, 0], index[:, 1], index[:, 2]] = True

    return dilation(volume, footprint=ball(radius_vox))


def voxel_graph_to_world(G, N):
    H = nx.MultiGraph(G)
    dx = 2*BOUND/(N-1)
    origin = np.array([-BOUND, -BOUND, -BOUND], dtype=float)

    for node, data in H.nodes(data=True):
        data["pos"] = origin + dx*np.asarray(data["pos"], dtype=float)
    for u, v, key, data in H.edges(keys=True, data=True):
        data["pts"] = origin + dx*np.asarray(data["pts"], dtype=float)

    return H


def recover_graph(volume, N):
    skeleton = skeletonize(volume, method="lee")
    raw = skeleton_image_to_graph(skeleton)
    world = voxel_graph_to_world(raw, N)
    return simplify_edges(world)


def yamada_result(G):
    result = compute_yamada_polynomial(
        G,
        A,
        rotation_angles=None,
        num_rotation_samples=PROJECTION_SAMPLES,
        crossing_warning_threshold=None,
        normalize=True,
        n_jobs=1,
        method="recursive",
        return_result=True,
    )
    return sp.expand(result.polynomial), result.projection


def same_polynomial(left, right):
    return sp.simplify(
        sp.together(sp.expand(left-right))
    ) == 0


## Graph-level pre-certification

Before voxelization, each active generating graph is tested under all allowed affine
deformations. This isolates the continuous geometry step from the discretization
step.

A failure here is **not** a voxel-recovery failure and aborts the benchmark with
the actual polynomials and crossing counts printed for diagnosis.


In [ ]:
TARGETS = {}
for case in CASES:
    polynomial, projection = yamada_result(case.graph)
    TARGETS[case.name] = polynomial
    print(
        f"TARGET {case.name:20s} "
        f"crossings={projection.num_crossings:2d} "
        f"Yamada={polynomial}"
    )

for case in CASES:
    for transform in TRANSFORMS:
        polynomial, projection = yamada_result(
            deform(case.graph, transform)
        )
        if not same_polynomial(polynomial, TARGETS[case.name]):
            raise AssertionError(
                f"{case.name}/{transform}: graph-level Yamada mismatch; "
                f"target={TARGETS[case.name]}, got={polynomial}, "
                f"selected crossings={projection.num_crossings}"
            )

print(
    "PASS: every active topology-preserving affine deformation "
    "preserves graph-level normalized Yamada."
)


## End-to-end voxel recovery

Only combinations passing the conservative continuous tube-thickness guard enter
the headline denominator. Each accepted case is voxelized, skeletonized, converted
back to an embedded graph, simplified, and evaluated by Yamada.

Results are checkpointed after every parameter point. The checkpoint includes a
configuration/code signature, so stale rows from an older notebook revision are
ignored rather than silently reused.


In [ ]:
def config_signature():
    payload = {
        "schema": CACHE_SCHEMA,
        "mode": "quick" if QUICK_MODE else "full",
        "cases": [case.name for case in CASES],
        "resolutions": RESOLUTIONS,
        "radii": TUBE_RADII_VOX,
        "transforms": TRANSFORMS,
        "projection_samples": PROJECTION_SAMPLES,
        "knotted_graph": str(kg_path),
    }
    return hashlib.sha256(
        json.dumps(payload, sort_keys=True).encode()
    ).hexdigest()[:20]


SIGNATURE = config_signature()


def result_key(case_name, transform, N, radius_vox):
    return (
        SIGNATURE,
        case_name,
        transform,
        int(N),
        int(radius_vox),
    )


done = {}
if RESUME and CHECKPOINT.exists():
    for line in CHECKPOINT.read_text().splitlines():
        if not line.strip():
            continue
        row = json.loads(line)
        if row.get("signature") != SIGNATURE:
            continue
        key = result_key(
            row["case"],
            row["transform"],
            row["resolution"],
            row["radius_vox"],
        )
        done[key] = row
    print("Resuming", len(done), "matching completed points.")


records = []
CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)

for case in CASES:
    for transform in TRANSFORMS:
        G = deform(case.graph, transform)

        for N in RESOLUTIONS:
            for radius_vox in TUBE_RADII_VOX:
                key = result_key(
                    case.name,
                    transform,
                    N,
                    radius_vox,
                )

                if key in done:
                    row = done[key]
                    records.append(row)
                    print("CACHED", key[1:])
                    continue

                allowed, radius_world, separation, limit = admissible(
                    case,
                    G,
                    N,
                    radius_vox,
                )

                row = {
                    "signature": SIGNATURE,
                    "case": case.name,
                    "transform": transform,
                    "resolution": int(N),
                    "radius_vox": int(radius_vox),
                    "radius_world": float(radius_world),
                    "separation": float(separation),
                    "limit": float(limit),
                    "admissible": bool(allowed),
                    "success": None,
                    "final_V": None,
                    "final_E": None,
                    "selected_crossings": None,
                    "recovered_yamada": None,
                    "error": None,
                }

                if allowed:
                    try:
                        recovered = recover_graph(
                            voxelize(G, N, radius_vox),
                            N,
                        )
                        recovered_poly, projection = yamada_result(
                            recovered
                        )
                        row.update(
                            success=bool(
                                same_polynomial(
                                    recovered_poly,
                                    TARGETS[case.name],
                                )
                            ),
                            final_V=int(recovered.number_of_nodes()),
                            final_E=int(recovered.number_of_edges()),
                            selected_crossings=int(
                                projection.num_crossings
                            ),
                            recovered_yamada=str(recovered_poly),
                        )
                    except Exception as exc:
                        row.update(
                            success=False,
                            error=f"{type(exc).__name__}: {exc}",
                        )

                records.append(row)

                with CHECKPOINT.open("a") as handle:
                    handle.write(json.dumps(row) + "\n")

                mark = (
                    "SKIP"
                    if not allowed
                    else ("PASS" if row["success"] else "FAIL")
                )
                print(
                    f"{mark:4s} {case.name:20s} {transform:8s} "
                    f"N={N:3d} r={radius_vox} "
                    f"V/E={row['final_V']}/{row['final_E']} "
                    f"x={row['selected_crossings']}"
                )


valid = [row for row in records if row["admissible"]]
passed = sum(bool(row["success"]) for row in valid)

if not valid:
    raise RuntimeError(
        "No admissible parameter point survived the tube-thickness guard."
    )

print(
    f"\nHeadline Yamada recovery: "
    f"{passed}/{len(valid)} = {100*passed/len(valid):.2f}%"
)

for row in valid:
    if not row["success"]:
        print(
            "FAILURE:",
            row["case"],
            row["transform"],
            "N=", row["resolution"],
            "r=", row["radius_vox"],
            row["error"] or (
                f"Yamada mismatch: {row['recovered_yamada']}"
            ),
        )

if QUICK_MODE and passed != len(valid):
    raise AssertionError(
        "QUICK_MODE is a correctness smoke test and must pass every "
        "admissible point before the full local sweep is trusted."
    )


In [ ]:
resolutions = RESOLUTIONS
radii = TUBE_RADII_VOX

heat = np.full(
    (len(radii), len(resolutions)),
    np.nan,
    dtype=float,
)

for i, radius_vox in enumerate(radii):
    for j, N in enumerate(resolutions):
        group = [
            row for row in records
            if row["admissible"]
            and row["radius_vox"] == radius_vox
            and row["resolution"] == N
        ]
        if group:
            heat[i, j] = np.mean(
                [bool(row["success"]) for row in group]
            )

fig, ax = plt.subplots(figsize=(8, 3.5))
im = ax.imshow(
    heat,
    vmin=0,
    vmax=1,
    origin="lower",
    aspect="auto",
)
ax.set_xticks(range(len(resolutions)), resolutions)
ax.set_yticks(range(len(radii)), radii)
ax.set_xlabel("voxel resolution N")
ax.set_ylabel("tube radius [voxels]")
ax.set_title("Yamada recovery rate")
fig.colorbar(im, ax=ax, label="recovery fraction")
plt.show()


names = [case.name for case in CASES]
rates = []
for name in names:
    group = [
        row for row in records
        if row["admissible"] and row["case"] == name
    ]
    rates.append(
        np.mean([bool(row["success"]) for row in group])
        if group else np.nan
    )

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(range(len(names)), rates)
ax.set_ylim(0, 1.05)
ax.set_xticks(
    range(len(names)),
    names,
    rotation=60,
    ha="right",
)
ax.set_ylabel("Yamada recovery fraction")
plt.tight_layout()
plt.show()


### Interpretation

- `admissible=False`: the requested continuous tube is already too thick relative
  to the conservative geometric clearance guard, so that point is excluded from
  the recovery denominator.
- `admissible=True, success=False`: a genuine end-to-end failure occurred after a
  topology-preserving deformation.
- `QUICK_MODE=True`: every admissible point must pass; otherwise the notebook
  raises and the full sweep should not be trusted yet.
- `QUICK_MODE=False`: the full \(N=150,175,\ldots,300\) sweep records successes and
  failures rather than requiring a perfect recovery rate. Those failures are the
  data needed for the benchmark's validity map.

The removed knot/link self-loop cases should be revisited separately after the
projection layer has an explicit robust representation for vertex-free closed
components.
